# 05B (Colab) — Train One Segmentation Model

Run on **Google Colab** with GPU runtime while Kaggle runs other models in parallel.

**Runtime → Change runtime type → T4 GPU**

Run once per model — change `CONFIG_NAME` each time.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
GITHUB_REPO  = 'https://github.com/yasmineelsayeddd/brain-tumor-segmentation.git'
DATASET_SLUG = 'yasmineelqorashy/brats2020-2d-prepared'
KAGGLE_TOKEN = 'KGAT_bb60250db72735c2a11893aa4a1e0db7'  # replace if expired

# Change this per run:
CONFIG_NAME    = 'default.yaml'
# CONFIG_NAME  = 'unet_best_fusion_loss_aug.yaml'
# CONFIG_NAME  = 'attention_unet.yaml'
# CONFIG_NAME  = 'uncertainty_unet.yaml'
# CONFIG_NAME  = 'unetpp.yaml'

EPOCH_OVERRIDE = None  # set to 2 for a quick smoke test

# Checkpoints are written straight to this Google Drive folder, every epoch.
# Re-running this notebook on ANY account auto-continues from the last epoch —
# nothing is lost when a free-tier session is cut off.
DRIVE_CKPT_DIR = '/content/drive/MyDrive/brats_checkpoints'

In [ ]:
import os, shutil, subprocess, sys, json, yaml
from pathlib import Path

# Install missing packages
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'kaggle', 'albumentations', 'pyyaml'])

os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 1. Clone code repo ─────────────────────────────────────────────────────
os.chdir('/content')   # reset cwd before deleting PROJECT

PROJECT = Path('/content/project')
if PROJECT.exists():
    shutil.rmtree(PROJECT)

subprocess.check_call(['git', 'clone', '--depth', '1', GITHUB_REPO, str(PROJECT)])
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

def run(cmd):
    env  = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    proc = subprocess.Popen(cmd, cwd=str(PROJECT), env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

print('Repo cloned to:', PROJECT)

In [ ]:
# ── 2. Download prepared data from Kaggle (~12 GB, ~5 min) ─────────────────
DATA_ROOT  = Path('/content/brats2020_2d')
SPLIT_FILE = Path('/content/splits/default.json')

if not (DATA_ROOT / 'metadata.json').exists():
    print('Downloading brats2020-2d-prepared (~12 GB) …')
    dl_dir = Path('/content/kaggle_dl')
    dl_dir.mkdir(exist_ok=True)
    subprocess.check_call([
        'kaggle', 'datasets', 'download',
        '-d', DATASET_SLUG,
        '-p', str(dl_dir),
        '--unzip',
    ])
    # Dataset structure: data/brats2020_2d/ and configs/splits/default.json
    extracted = next(dl_dir.iterdir()) if len(list(dl_dir.iterdir())) == 1 and next(dl_dir.iterdir()).is_dir() else dl_dir
    src_data  = next(extracted.rglob('metadata.json')).parent
    src_split = next(extracted.rglob('default.json'))
    shutil.copytree(src_data, DATA_ROOT)
    SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_split, SPLIT_FILE)
    shutil.rmtree(dl_dir)
    print('Done.')
else:
    print('Data already present.')

with open(DATA_ROOT / 'metadata.json') as f:
    meta = json.load(f)
with open(SPLIT_FILE) as f:
    split = json.load(f)
print(f'Slices: {meta["num_slices"]:,}  |  Split: { {k: len(v) for k, v in split.items()} }')

In [ ]:
# ── 3. Mount Drive + build runtime config (checkpoints go straight to Drive) ─
from google.colab import drive
drive.mount('/content/drive')

ckpt_dir = Path(DRIVE_CKPT_DIR)
ckpt_dir.mkdir(parents=True, exist_ok=True)

with open(PROJECT / 'configs' / CONFIG_NAME) as f:
    cfg = yaml.safe_load(f)

cfg['data']['data_root']        = str(DATA_ROOT)
cfg['data']['split_file']       = str(SPLIT_FILE)
cfg['checkpoint_dir']           = str(ckpt_dir)        # written to Drive every epoch
cfg['experiment']['output_dir'] = '/content/outputs'

if EPOCH_OVERRIDE is not None:
    cfg['training']['epochs'] = int(EPOCH_OVERRIDE)
    cfg['training']['early_stopping_patience'] = max(2, min(int(EPOCH_OVERRIDE), 15))

runtime_cfg = Path('/content/runtime_config.yaml')
with open(runtime_cfg, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

exp_name = cfg['experiment']['name']
print(f'Model:       {cfg["model"]["arch"]}')
print(f'Epochs:      {cfg["training"]["epochs"]}')
print(f'Exp:         {exp_name}')
print(f'Checkpoints: {ckpt_dir}')

In [ ]:
# ── 4. Train (auto-resumes if a checkpoint for this model exists on Drive) ──
last_ckpt = ckpt_dir / f'{exp_name}_last.pth'
best_ckpt = ckpt_dir / f'{exp_name}_best.pth'

resume_arg = []
if last_ckpt.exists():
    resume_arg = ['--resume', str(last_ckpt)]
    print(f'Resuming from {last_ckpt.name}')
elif best_ckpt.exists():
    resume_arg = ['--resume', str(best_ckpt)]
    print(f'Resuming from {best_ckpt.name} (first resume — no _last.pth yet)')
else:
    print(f'No checkpoint for {exp_name} — training from scratch')

run([sys.executable, '-m', 'scripts.train', '--config', str(runtime_cfg)] + resume_arg)

In [ ]:
# ── 5. Training curves ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

exp_name  = cfg['experiment']['name']
hist_path = Path('/content/outputs') / exp_name / 'history.json'

if hist_path.exists():
    with open(hist_path) as f:
        history = json.load(f)
    epochs     = [h['epoch']      for h in history]
    train_dice = [h['train_dice'] for h in history]
    val_dice   = [h['val_dice']   for h in history]
    fig, axes  = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_dice, label='train')
    axes[0].plot(epochs, val_dice,   label='val')
    axes[0].set_title(f'{exp_name} — Dice'); axes[0].legend()
    axes[1].plot(epochs, [h['val_loss'] for h in history], color='orange')
    axes[1].set_title('Val Loss')
    plt.tight_layout(); plt.show()
    best = max(history, key=lambda h: h['val_dice'])
    print(f'Best: epoch {best["epoch"]}  val_dice={best["val_dice"]:.4f}')

In [ ]:
# ── 6. (Optional) Download checkpoint to your machine ──────────────────────
# NOT required — the checkpoint already lives safely in your Google Drive
# (brats_checkpoints/). Run this only if you also want a local copy.
from google.colab import files

ckpt = ckpt_dir / f'{exp_name}_best.pth'
print(f'Checkpoint: {ckpt}  exists={ckpt.exists()}')
if ckpt.exists():
    print(f'Size: {ckpt.stat().st_size / 1e6:.1f} MB  — already saved in Drive')
    files.download(str(ckpt))